In [1]:
import os
import sys

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np

import xesmf as xe
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
# settings
%config InlineBackend.figure_format = 'retina'

# top level directories
dpath0='/Users/dervlamk/OneDrive/research/topo_nam'
dpath1= f'{dpath0}/topo_files'
# save figs here
opath='/Users/dervlamk/OneDrive/research/topo_nam/figs'

ModuleNotFoundError: No module named 'ESMF'

In [ ]:
import sys
print(sys.executable)

In [ ]:
### +++ IMPORT DATA +++ ###

# initialize dicts
flor = {}
hrmip = {}

# ETOPO05
etopo_full=xr.open_dataset(f'{dpath1}/obs.etopo5.zsurf.nc').ROSE
etopo_full=etopo_full.rename({'ETOPO05_Y':'lat', 'ETOPO05_X':'lon'}) # rename dimensions 
etopo=etopo_full.where(etopo_full>0,np.nan)  # crudely remove bathymetry

# CM2.5-FLOR
flor_mask = xr.open_dataset(f'{dpath1}/flor.land_mask.nc').LAND_MASK.rename({'lat':'GRID_YT', 'lon':'GRID_XT'})

flor['ctrl'] = xr.open_dataset(f'{dpath1}/flor.ctrl.zsurf.nc').ZSURF.where(flor_mask>0.5,np.nan)
flor['ctrl'] = flor['ctrl'].rename({'GRID_YT':'lat', 'GRID_XT':'lon'})

flor['hicam'] = xr.open_dataset(f'{dpath1}/flor.cam.zsurf.nc').ZSURF.where(flor_mask>0.5,np.nan)
flor['hicam'] = flor['hicam'].rename({'GRID_YT':'lat', 'GRID_XT':'lon'})

flor['hitopo'] = xr.open_dataset(f'{dpath1}/flor.hitopo.zsurf.nc').ZSURF.where(flor_mask>0.5,np.nan)
flor['hitopo'] = flor['hitopo'].rename({'GRID_YT':'lat', 'GRID_XT':'lon'})

flor['hicam-diff'] = flor['hicam'] - flor['ctrl']
flor['hitopo-diff'] = flor['hitopo'] - flor['ctrl']

# HighResMIP models
cmcc_hr_mask = xr.open_dataset(f'{dpath1}/sftlf_fx_CMCC-CM2-HR4_highresSST-present_r1i1p1f1_gn.nc').sftlf/100
cmcc_vhr_mask = xr.open_dataset(f'{dpath1}/sftlf_fx_CMCC-CM2-VHR4_highresSST-present_r1i1p1f1_gn.nc').sftlf/100

hrmip['low'] = xr.open_dataset(f'{dpath1}/orog_fx_CMCC-CM2-HR4_highresSST-present_r1i1p1f1_gn.nc').orog.where(cmcc_hr_mask>0.5,np.nan)
hrmip['high'] = xr.open_dataset(f'{dpath1}/orog_fx_CMCC-CM2-VHR4_highresSST-present_r1i1p1f1_gn.nc').orog.where(cmcc_vhr_mask>0.5,np.nan)


In [ ]:
# calculate meridional maximum surface height for longitudinal transect spanning Mexico

topo_profile = {}

# set lat/lon bounds
ymin=24
ymax=29
xmin=245
xmax=265

# extract  
topo_profile['etopo']  = etopo.sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
topo_profile['ctrl']   = flor['ctrl'].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
topo_profile['hicam']  = flor['hicam'].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
topo_profile['hitopo'] = flor['hitopo'].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
topo_profile['hrmip-low'] = hrmip['low'].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
topo_profile['hrmip-hi'] = hrmip['high'].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)


In [ ]:
# calculate meridional maximum surface height for longitudinal transect spanning Mexico

northern_topo_profile = {}

# set lat/lon bounds
ymin=31.5
ymax=33.5
xmin=241
xmax=260


# extract  
northern_topo_profile['etopo']  = etopo.sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
northern_topo_profile['ctrl']   = flor['ctrl'].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
northern_topo_profile['hicam']  = flor['hicam'].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
northern_topo_profile['hitopo'] = flor['hitopo'].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
northern_topo_profile['hrmip-low'] = hrmip['low'].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
northern_topo_profile['hrmip-hi'] = hrmip['high'].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)


In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# label specs
text_kw={'size':16, 'weight':'bold', 'color':'k', 'ha':'left', 'va':'center'}
text_kw1={'size':16, 'weight':'bold', 'color':'k', 'ha':'center', 'va':'bottom'}
text_kw2={'size':22, 'weight':'bold', 'color':'k', 'ha':'center', 'va':'center'}
labels=['ETOPO05',
        'CM2.5-FLOR CTRL',
        r'CM2.5-FLOR HI-MEX',
        r'CM2.5-FLOR HI-GBL',
        'CMCC-CM2-HR4',
        'CMCC-CM2-VHR4']
titles=['ETOPO05',
        'CMCC-CM2-HR4',
        'CMCC-CM2-VHR4',
        'CM2.5-FLOR CTRL',
        r'HI-MEX$-$CTRL',
        r'HI-GBL$-$CTRL']
letters=['A','B','C','D','E','F','G','H']
tx=-105
ty=40
# patch specs
patch_kw = {'ec':'dodgerblue', 'lw':2, 'ls':'--', 'fc':'none', 'clip_on':False, 'zorder':100}
# for line plot
tkw = {'axis': 'both', 'direction':'in', 'labelsize': 'x-large'} 
arrow_kw=dict(color='black', linewidth=.15, shrink=0)
arrow_text_kw={'size':12, 'weight':'normal', 'color':'k', 'ha':'center', 'va':'bottom'}
legend_prop={'size':9, 'weight':'bold'}
legend_kw={'loc':'upper right', 'bbox_to_anchor':(1.18, .95),
           'labelcolor':'linecolor', 'ncols':1, 'frameon':False}
colors=['k','grey','coral','orangered','goldenrod','peru'] #'thistle','orchid']
steploc='mid'
# colormap -- topo field
cmap=cm.afmhot
vmin=0
vmax=3400
levels=np.linspace(vmin, vmax, 18)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
# colormap -- topo difference
#dcmap=combine_cmaps_white_center(cm.YlOrBr, cm.bone, range_low=[1,0.1], range_up=[1,0], n_low=128, n_up=128, n_white=18)
dcmap=cm.twilight_shifted
dvmin=-1000
dvmax=1000
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -90., 15., 40.]


fig = plt.figure(figsize=(15, 14))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.2, wspace=0.2)

# map subplots
ax1 = fig.add_subplot(gs[0, 0], projection=proj)
ax1.pcolormesh(etopo.lon, etopo.lat, etopo, cmap=cmap, norm=norm, transform=trans)

ax2 = fig.add_subplot(gs[0, 1], projection=proj)
ax2.pcolormesh(hrmip['low'].lon, hrmip['low'].lat, hrmip['low'], cmap=cmap, norm=norm, transform=trans)

ax3 = fig.add_subplot(gs[0, 2], projection=proj)
ax3.pcolormesh(hrmip['high'].lon, hrmip['high'].lat, hrmip['high'], cmap=cmap, norm=norm, transform=trans)

ax4 = fig.add_subplot(gs[1, 0], projection=proj)
cf=ax4.pcolormesh(flor['ctrl'].lon, flor['ctrl'].lat, flor['ctrl'], cmap=cmap, norm=norm, transform=trans)

ax5 = fig.add_subplot(gs[1, 1], projection=proj)
ax5.pcolormesh(flor['hicam'].lon, flor['hicam'].lat, flor['hicam-diff'], cmap=dcmap, norm=dnorm, transform=trans)

ax6 = fig.add_subplot(gs[1, 2], projection=proj)
cf2=ax6.pcolormesh(flor['hitopo'].lon, flor['hitopo'].lat, flor['hitopo-diff'], cmap=dcmap, norm=dnorm, transform=trans)

for i,ax in enumerate([ax1, ax2, ax3, ax4, ax5, ax6]):
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw1)
    ax.text(-120.5,ty+1.5,letters[i], **text_kw2, zorder=100)
    # add box around region of topographic profile
    pp=plt.Rectangle((-115, 24), 20, 4, label='_Hidden', **patch_kw) 
    ax.add_patch(pp)
    # map properties
    ax.coastlines(color='k')
    #ax.add_feature(cfeature.OCEAN, fc='w', zorder=10)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':12}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':12}

# add colorbar
cax=fig.add_axes([.92, .67, 0.02, 0.2])
cbar=fig.colorbar(cf, orientation='vertical', extend='max', cax=cax)
cbar.set_ticks([0,1000,2000,3000])
cbar.set_ticklabels([0.0,1.0,2.0,3.0])
cbar.set_label('Surface Elevation [km]', labelpad=30, rotation=270, size=16, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=16)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')
    
cax=fig.add_axes([.92, .395, 0.02, 0.2])
cbar=fig.colorbar(cf2, orientation='vertical', extend='both', cax=cax)
cbar.set_ticks([-1000, -500, 0, 500, 1000])
cbar.set_ticklabels([-1.0, -0.5 ,0.0, 0.5, 1.0])
cbar.set_label('$\Delta$ Surface Elevation [km]', labelpad=30, rotation=270, size=16, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=16)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')
    
    

# topography profiles
ax7 = fig.add_subplot(gs[2, :])
ax7.text(245.5, 3750, f'Meridional Maximum Surface Height [{ymin}°N$-${ymax}°N]', rotation=0, **text_kw)
ax7.text(244.75, 3800, letters[6], **text_kw2)

# plot topography
for i,(case,profile) in enumerate(topo_profile.items()):
    lcol = colors[i]
    if case in ['ctrl']:
        ls='--'
    elif case in ['hrmip-low']:
        ls='dotted'
    else:
        ls='-'
    ax7.step(topo_profile[case].lon, profile, where=steploc, color=lcol, lw=2, ls=ls, label=labels[i])

ax7.annotate('Pacific\nOcean', xy=(245.5,2750), xytext=(245.5,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax7.annotate('Baja\nPeninsula', xy=(247,2750), xytext=(247,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax7.annotate('Gulf of\nCalifornia', xy=(249,2750), xytext=(249,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax7.annotate('Sierra Madre\nOccidental', xy=(252.3,2750), xytext=(252.3,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax7.annotate('Mexican\nAltiplano', xy=(256.5,2750), xytext=(256.5,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax7.annotate('Sierra Madre\nOriental', xy=(259.7,2750), xytext=(259.7,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax7.annotate('Gulf of\nMexico', xy=(264,2750), xytext=(264,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)

# plot properties
ax7.set(xlim=[245, 265], ylim=[0,3600])
ax7.set_ylabel('SURFACE ELEVATION [km]', fontsize=14, fontweight='normal')
ax7.yaxis.set_tick_params(labelsize='x-large', color='k', labelcolor='k')
ax7.set_xlabel('LONGITUDE [°E]', fontsize=14, fontweight='normal')
ax7.xaxis.set_tick_params(labelsize='x-large', color='k', labelcolor='k')
ax7.set_yticks([500,1000,1500,2000,2500,3000,3500])
ax7.set_yticklabels([0.5,1.0,1.5,2.0,2.5,3.0,3.5])
ax7.tick_params(**tkw)
ax7.legend(prop=legend_prop, **legend_kw)

plt.savefig(f'{opath}/fig1.topo-comparison.png', transparent=True, bbox_inches='tight')
plt.savefig(f'{opath}/fig1.topo-comparison.pdf', transparent=True, bbox_inches='tight')

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# label specs
text_kw={'size':16, 'weight':'bold', 'color':'k', 'ha':'left', 'va':'center'}
text_kw1={'size':16, 'weight':'bold', 'color':'k', 'ha':'center', 'va':'bottom'}
text_kw2={'size':22, 'weight':'bold', 'color':'k', 'ha':'center', 'va':'center'}
labels=['ETOPO05',
        'CM2.5-FLOR CTRL',
        r'CM2.5-FLOR HI-MEX',
        r'CM2.5-FLOR HI-GBL',
        'CMCC-CM2-HR4',
        'CMCC-CM2-VHR4']
titles=['ETOPO05',
        'CMCC-CM2-HR4',
        'CMCC-CM2-VHR4',
        'CM2.5-FLOR CTRL',
        r'HI-MEX$-$CTRL',
        r'HI-GBL$-$CTRL']
letters=['A','B','C','D','E','F','G','H','I','J']
tx=-105
ty=40
# patch specs
patch_kw = {'ec':'dodgerblue', 'lw':2, 'ls':'--', 'fc':'none', 'clip_on':False, 'zorder':100}
# for line plot
tkw = {'axis': 'both', 'direction':'in', 'labelsize': 'x-large'} 
arrow_kw=dict(color='black', linewidth=.15, shrink=0)
arrow_text_kw={'size':12, 'weight':'normal', 'color':'k', 'ha':'center', 'va':'bottom'}
legend_prop={'size':9, 'weight':'bold'}
legend_kw={'loc':'upper right', 'bbox_to_anchor':(1, .95),
           'labelcolor':'linecolor', 'ncols':1, 'frameon':False}
colors=['k','grey','coral','orangered','goldenrod','peru']
steploc='mid'
# colormap -- topo field
cmap=cm.afmhot
vmin=0
vmax=3400
levels=np.linspace(vmin, vmax, 18)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
# colormap -- topo difference
#dcmap=combine_cmaps_white_center(cm.YlOrBr, cm.bone, range_low=[1,0.1], range_up=[1,0], n_low=128, n_up=128, n_white=18)
dcmap=cm.twilight_shifted
dvmin=-1000
dvmax=1000
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -90., 15., 40.]


fig = plt.figure(figsize=(15, 18))
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.2, wspace=0.25)

# map subplots
ax1 = fig.add_subplot(gs[0, 0], projection=proj)
ax1.pcolormesh(etopo.lon, etopo.lat, etopo, cmap=cmap, norm=norm, transform=trans)

ax2 = fig.add_subplot(gs[0, 1], projection=proj)
ax2.pcolormesh(hrmip['low'].lon, hrmip['low'].lat, hrmip['low'], cmap=cmap, norm=norm, transform=trans)

ax3 = fig.add_subplot(gs[0, 2], projection=proj)
ax3.pcolormesh(hrmip['high'].lon, hrmip['high'].lat, hrmip['high'], cmap=cmap, norm=norm, transform=trans)

ax4 = fig.add_subplot(gs[1, 0], projection=proj)
cf=ax4.pcolormesh(flor['ctrl'].lon, flor['ctrl'].lat, flor['ctrl'], cmap=cmap, norm=norm, transform=trans)

ax5 = fig.add_subplot(gs[1, 1], projection=proj)
ax5.pcolormesh(flor['hicam'].lon, flor['hicam'].lat, flor['hicam-diff'], cmap=dcmap, norm=dnorm, transform=trans)

ax6 = fig.add_subplot(gs[1, 2], projection=proj)
cf2=ax6.pcolormesh(flor['hitopo'].lon, flor['hitopo'].lat, flor['hitopo-diff'], cmap=dcmap, norm=dnorm, transform=trans)

for i,ax in enumerate([ax1, ax2, ax3, ax4, ax5, ax6]):
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw1)
    ax.text(-120.5,ty+1.5,letters[i], **text_kw2, zorder=100)
    # add box around region of topographic profile
    pp2=plt.Rectangle((-119, 31.5), 19, 2, label='_Hidden', **patch_kw) 
    ax.add_patch(pp2)
    pp=plt.Rectangle((-115, 24), 20, 4, label='_Hidden', **patch_kw) 
    ax.add_patch(pp)
    # map properties
    ax.coastlines(color='k')
    #ax.add_feature(cfeature.OCEAN, fc='w', zorder=10)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':12}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':12}

# add colorbar
cax=fig.add_axes([.92, .725, 0.02, 0.15])
cbar=fig.colorbar(cf, orientation='vertical', extend='max', cax=cax)
cbar.set_ticks([0,1000,2000,3000])
cbar.set_ticklabels([0.0,1.0,2.0,3.0])
cbar.set_label('Surface Elevation [km]', labelpad=30, rotation=270, size=16, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=16)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')
    
cax=fig.add_axes([.92, .52, 0.02, 0.15])
cbar=fig.colorbar(cf2, orientation='vertical', extend='both', cax=cax)
cbar.set_ticks([-1000, -500, 0, 500, 1000])
cbar.set_ticklabels([-1.0, -0.5 ,0.0, 0.5, 1.0])
cbar.set_label('$\Delta$ Surface Elevation [km]', labelpad=30, rotation=270, size=16, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=16)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')
    
    

# topography profiles
ax7 = fig.add_subplot(gs[2, :])
ax7.text(241.5, 3350, f'Meridional Maximum Surface Height [31.5°N$-$33.5°N]', rotation=0, **text_kw)
ax7.text(240.75, 3400, letters[6], **text_kw2)

# plot topography
for i,(case,profile) in enumerate(northern_topo_profile.items()):
    lcol = colors[i]
    if case in ['ctrl']:
        ls='--'
    elif case in ['hrmip-low']:
        ls='dotted'
    else:
        ls='-'
    ax7.step(northern_topo_profile[case].lon, profile, where=steploc, color=lcol, lw=2, ls=ls, label=labels[i])

ax7.annotate('Pacific\nOcean', xy=(242,2400), xytext=(242,2750), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax7.annotate('Baja\nPeninsula', xy=(243.75,2400), xytext=(243.75,2750), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax7.annotate('Low\nDeserts', xy=(246,2400), xytext=(246,2750), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax7.annotate('Sierra Madre\nOccidental', xy=(250,2400), xytext=(250,2750), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)

# plot properties
ax7.set(xlim=[241, 260], ylim=[0,3200])
ax7.set_ylabel('SURFACE ELEVATION [km]', fontsize=14, fontweight='normal')
ax7.yaxis.set_tick_params(labelsize='x-large', color='k', labelcolor='k')
#ax7.set_xlabel('LONGITUDE [°E]', fontsize=14, fontweight='normal')
ax7.xaxis.set_tick_params(labelsize='x-large', color='k', labelcolor='k')
ax7.set_yticks([500,1000,1500,2000,2500,3000])
ax7.set_yticklabels([0.5,1.0,1.5,2.0,2.5,3.0])
ax7.tick_params(**tkw)
ax7.legend(prop=legend_prop, **legend_kw)



ax8 = fig.add_subplot(gs[3, :])
ax8.text(245.5, 3750, f'Meridional Maximum Surface Height [24°N$-$28°N]', rotation=0, **text_kw)
ax8.text(244.75, 3800, letters[7], **text_kw2)

# plot topography
for i,(case,profile) in enumerate(topo_profile.items()):
    lcol = colors[i]
    if case in ['ctrl']:
        ls='--'
    elif case in ['hrmip-low']:
        ls='dotted'
    else:
        ls='-'
    ax8.step(topo_profile[case].lon, profile, where=steploc, color=lcol, lw=2, ls=ls, label=labels[i])

ax8.annotate('Pacific\nOcean', xy=(245.5,2750), xytext=(245.5,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax8.annotate('Baja\nPeninsula', xy=(247,2750), xytext=(247,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax8.annotate('Gulf of\nCalifornia', xy=(249,2750), xytext=(249,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax8.annotate('Sierra Madre\nOccidental', xy=(252.3,2750), xytext=(252.3,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax8.annotate('Mexican\nAltiplano', xy=(256.5,2750), xytext=(256.5,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax8.annotate('Sierra Madre\nOriental', xy=(259.7,2750), xytext=(259.7,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)
ax8.annotate('Gulf of\nMexico', xy=(264,2750), xytext=(264,3100), label='_Hidden', arrowprops=arrow_kw, **arrow_text_kw)

# plot properties
ax8.set(xlim=[245, 265], ylim=[0,3600])
ax8.set_ylabel('SURFACE ELEVATION [km]', fontsize=14, fontweight='normal')
ax8.yaxis.set_tick_params(labelsize='x-large', color='k', labelcolor='k')
ax8.set_xlabel('LONGITUDE [°E]', fontsize=14, fontweight='normal')
ax8.xaxis.set_tick_params(labelsize='x-large', color='k', labelcolor='k')
ax8.set_yticks([500,1000,1500,2000,2500,3000,3500])
ax8.set_yticklabels([0.5,1.0,1.5,2.0,2.5,3.0,3.5])
ax8.tick_params(**tkw)
#ax8.legend(prop=legend_prop, **legend_kw)


plt.savefig(f'{opath}/fig1.topo-comparison.v2.png', transparent=True, bbox_inches='tight')
plt.savefig(f'{opath}/fig1.topo-comparison.v2.pdf', transparent=True, bbox_inches='tight')

## Profiles exemplifying SMO height underestimation

In [ ]:
# interpolate etopo05 to 100 km and 50 km grid

# Compute target number of points
# Rough approximation: original degrees / factor
lat_fine = etopo.lat
lon_fine = etopo.lon

lat_coarse = {}
lon_coarse = {}
# 100 km ~ 1°
lat_coarse['100km'] = np.arange(lat_fine.min(), lat_fine.max()+0.01, 1.0)
lon_coarse['100km'] = np.arange(lon_fine.min(), lon_fine.max()+0.01, 1.0)
# 50 km ~ 0.5°
lat_coarse['50km'] = np.arange(lat_fine.min(), lat_fine.max()+0.01, 0.5)
lon_coarse['50km'] = np.arange(lon_fine.min(), lon_fine.max()+0.01, 0.5)

etopo_coarse = {}
for res in ['100km','50km']
    coarse_grid = xr.Dataset(
        {
            'lat': (['lat'], lat_coarse[res]),
            'lon': (['lon'], lon_coarse[res])
        }
    )

    # Create regridder
    regridder = xe.Regridder(
        etopo,                 # original fine grid
        coarse_grid,        # target coarse grid
        method='conservative',  # preserves area-integrated quantities
        periodic=True       # if longitude wraps around (0–360)
    )

    etopo_coarse[res] = regridder(etopo)


In [ ]:
# calculate zonal maximum surface height for latitudinal transect spanning SMO

smo_profile = {}

# set lat/lon bounds
ymin=19
ymax=31
xmin=250
xmax=256

# extract  
smo_profile['etopo']  = etopo.sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)).max(dim="lat",skipna=True)
